# 🏷️ **Complete Guide: Categorical & Ordinal Data Handling in Python**

**Author:** Prashant Nair | VNex (Vihaan NextGen Solutions)  
**Dataset:** Employee Attrition & Performance Dataset (150 records)

---

## 📋 Table of Contents

1. **Understanding Categorical Data** — Definitions, Types, and Why It Matters  
2. **Dataset Overview & EDA** — Loading, profiling, and understanding our features  
3. **Handling Nominal Categorical Data**
   - 3.1 Label Encoding (and why it's risky for nominal)
   - 3.2 One-Hot Encoding (pd.get_dummies & OneHotEncoder)
   - 3.3 Binary Encoding
   - 3.4 Frequency / Count Encoding
   - 3.5 Target (Mean) Encoding
   - 3.6 Leave-One-Out Encoding
   - 3.7 Hashing Encoding
   - 3.8 BaseN Encoding
   - 3.9 James-Stein Encoding
4. **Handling Ordinal Categorical Data**
   - 4.1 OrdinalEncoder (sklearn)
   - 4.2 Pandas Categorical with ordered=True
   - 4.3 Manual Mapping (dict-based)
   - 4.4 Label Encoding (appropriate for ordinal)
5. **Handling Binary Categorical Data**
6. **Handling Missing Values in Categorical Columns**
7. **Comparison Matrix — When to Use What**
8. **Full Pipeline Example with sklearn ColumnTransformer**
9. **Key Takeaways & Best Practices**

---

## 1. Understanding Categorical Data

### What is Categorical Data?
Categorical data represents **discrete groups or labels** — not continuous numbers. Machine learning models (especially tree-based and linear models) need numerical inputs, so we must **encode** these categories.

### Three Types of Categorical Data

| Type | Definition | Example | Has Order? |
|------|-----------|---------|------------|
| **Nominal** | Categories with NO inherent order | City, Blood Group, Department | ❌ No |
| **Ordinal** | Categories WITH a meaningful order | Education Level (High School < Bachelor < Master < PhD) | ✅ Yes |
| **Binary** | Special case — only 2 categories | Yes/No, Male/Female, True/False | ❌/✅ Depends |

### Why Does the Encoding Method Matter?

- **Wrong encoding can introduce fake relationships.** If you assign City: Mumbai=1, Pune=2, Delhi=3, the model may infer Delhi > Pune > Mumbai — which is meaningless.
- **Right encoding preserves the true nature** of the data — whether it's unordered (nominal) or ordered (ordinal).
- **Dimensionality explosion**: One-Hot Encoding on a feature with 1000 categories creates 1000 new columns. Alternative methods like Hashing or Target Encoding solve this.

## 2. Dataset Overview & EDA

In [ ]:
# Install category_encoders (not available by default in Colab)
!pip install category_encoders -q

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Sklearn
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Category Encoders library
import category_encoders as ce

print('All libraries loaded successfully!')

In [ ]:
# Load the dataset
# Upload employee_data.csv to your Colab session or mount Google Drive
df = pd.read_csv('employee_data.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
# Data types and missing values
print('='*60)
print('DATA TYPES')
print('='*60)
print(df.dtypes)
print()
print('='*60)
print('MISSING VALUES')
print('='*60)
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Classify our columns
nominal_cols = ['Gender', 'City', 'Department', 'Employment_Type',
                'Blood_Group', 'Marital_Status']
ordinal_cols = ['Education_Level', 'Experience_Band',
                'Performance_Rating', 'Satisfaction_Level', 'Risk_Appetite']
binary_cols = ['Remote_Work', 'Attrition']
numerical_cols = ['Age', 'Salary', 'Years_at_Company', 'Training_Hours']
id_col = ['Employee_ID']

print(f'Nominal features ({len(nominal_cols)}): {nominal_cols}')
print(f'Ordinal features ({len(ordinal_cols)}): {ordinal_cols}')
print(f'Binary features ({len(binary_cols)}): {binary_cols}')
print(f'Numerical features ({len(numerical_cols)}): {numerical_cols}')

In [ ]:
# Unique value counts for all categorical features
print('CARDINALITY (Unique Values per Feature)')
print('-'*45)
for col in nominal_cols + ordinal_cols + binary_cols:
    print(f'{col:25s} -> {df[col].nunique():3d} unique values')
    print(f'{"":25s}    {list(df[col].dropna().unique())}')
    print()

In [ ]:
# Quick visual: Distribution of key categorical features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Key Categorical Features', fontsize=16, fontweight='bold')

for ax, col in zip(axes.flatten(), ['Department', 'Education_Level', 'Performance_Rating',
                                      'City', 'Satisfaction_Level', 'Attrition']):
    df[col].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('viridis', df[col].nunique()))
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 3. Handling Nominal Categorical Data

Nominal data has **NO inherent order**. Examples: City, Department, Blood Group.

> **Golden Rule:** Never use a method that introduces artificial ordering for nominal features.

### 3.1 Label Encoding (⚠️ Risky for Nominal Data)

**What it does:** Assigns an integer (0, 1, 2, ...) to each category alphabetically.

**When to use:**  
- ✅ Ordinal data (where order matters)  
- ✅ Target variable encoding  
- ✅ Tree-based models (Decision Trees, Random Forest, XGBoost) CAN handle label-encoded nominal data because they split on thresholds, but it's still not ideal.

**When NOT to use:**  
- ❌ Nominal data with linear models (Linear Regression, Logistic Regression, SVM) — the model will interpret the numbers as having mathematical meaning (3 > 2 > 1).

**Risk:** Mumbai=0, Pune=2, Delhi=1 implies Delhi is "between" Mumbai and Pune — which is meaningless.

In [ ]:
# Label Encoding — Demonstration (NOT recommended for nominal with linear models)
le = LabelEncoder()

# Encode the 'City' column
df_demo = df[['City']].copy()
df_demo['City_LabelEncoded'] = le.fit_transform(df_demo['City'])

print('Label Encoding of City:')
print('-'*40)
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for city, code in mapping.items():
    print(f'  {city:15s} -> {code}')

print()
df_demo.head(10)

⚠️ **Notice the problem:** Bangalore=0, Chennai=1, Delhi=2, Hyderabad=3... This implies Delhi > Chennai > Bangalore, which is completely meaningless for cities!

### 3.2 One-Hot Encoding (OHE) — The Gold Standard for Nominal Data

**What it does:** Creates a new binary (0/1) column for each category.

**When to use:**  
- ✅ Nominal features with **low to medium cardinality** (2-15 unique values)
- ✅ Any model type — linear, tree-based, neural networks
- ✅ When you need full interpretability

**When NOT to use:**  
- ❌ High cardinality features (100+ unique values) — creates too many columns (curse of dimensionality)
- ❌ When memory/computation is a constraint

**Variants:**  
- `drop_first=True` → drops one column to avoid **multicollinearity** (essential for linear/logistic regression)
- `drop_first=False` → keeps all columns (fine for tree-based models)

In [ ]:
# Method A: pd.get_dummies (Quick & easy, works on DataFrames directly)
df_ohe = pd.get_dummies(df[['City', 'Department']], prefix=['City', 'Dept'])

print(f'Original columns: City (7 unique), Department (8 unique)')
print(f'After OHE: {df_ohe.shape[1]} columns')
print(f'Column names: {list(df_ohe.columns)}')
print()
df_ohe.head()

In [ ]:
# Method A with drop_first=True (avoids multicollinearity)
df_ohe_dropped = pd.get_dummies(df[['City', 'Department']], prefix=['City', 'Dept'], drop_first=True)

print(f'With drop_first=True: {df_ohe_dropped.shape[1]} columns (reduced from {df_ohe.shape[1]})')
print(f'Dropped columns: City_Bangalore (reference) and Dept_Customer Support (reference)')
print()
df_ohe_dropped.head()

In [ ]:
# Method B: sklearn OneHotEncoder (Better for ML pipelines, handles unseen categories)
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='infrequent_if_exist')

city_encoded = ohe.fit_transform(df[['City']])
city_ohe_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(['City']))

print('sklearn OneHotEncoder with drop=first:')
city_ohe_df.head()

### 3.3 Binary Encoding

**What it does:** First applies Label Encoding, then converts the integer to its binary representation. Each binary digit becomes a separate column.

**When to use:**  
- ✅ **Medium-to-high cardinality** nominal features (10-100+ categories)  
- ✅ When One-Hot creates too many columns but you still want to avoid ordinal assumptions  
- ✅ Good balance between OHE and Label Encoding

**When NOT to use:**  
- ❌ Very low cardinality (2-5 categories) — OHE is simpler and equally effective

**Columns created:** log₂(n) columns instead of n columns.  
Example: 8 cities → OHE creates 8 columns, Binary Encoding creates only 4 columns.

In [ ]:
# Binary Encoding using category_encoders
binary_enc = ce.BinaryEncoder(cols=['City'])
df_binary = binary_enc.fit_transform(df[['City']])

print(f'Original: 1 column (City with {df["City"].nunique()} unique values)')
print(f'After Binary Encoding: {df_binary.shape[1]} columns')
print(f'Columns: {list(df_binary.columns)}')
print()
print('How it works:')
print('  Bangalore (1) -> 001 -> [0, 0, 1]')
print('  Chennai   (2) -> 010 -> [0, 1, 0]')
print('  Delhi     (3) -> 011 -> [0, 1, 1]')
print()
df_binary.head(10)

### 3.4 Frequency / Count Encoding

**What it does:** Replaces each category with its **frequency** (count or proportion) in the dataset.

**When to use:**  
- ✅ When the frequency of a category is itself informative (e.g., popular departments might have different attrition patterns)  
- ✅ High cardinality features  
- ✅ Tree-based models

**When NOT to use:**  
- ❌ When multiple categories have the **same frequency** (they'll get the same encoded value = information loss)  
- ❌ When frequency has no relationship with the target

In [ ]:
# Frequency Encoding — Manual Implementation
df_freq = df[['Department']].copy()

# Count Encoding
freq_map = df['Department'].value_counts().to_dict()
df_freq['Dept_CountEncoded'] = df['Department'].map(freq_map)

# Proportion Encoding (normalized)
prop_map = (df['Department'].value_counts(normalize=True)).to_dict()
df_freq['Dept_ProportionEncoded'] = df['Department'].map(prop_map).round(4)

print('Frequency / Count Encoding:')
print('-'*55)
for dept, count in sorted(freq_map.items(), key=lambda x: -x[1]):
    print(f'  {dept:20s} -> Count: {count:3d}  |  Proportion: {prop_map[dept]:.4f}')

print()
df_freq.head(10)

### 3.5 Target (Mean) Encoding

**What it does:** Replaces each category with the **mean of the target variable** for that category.

**When to use:**  
- ✅ High cardinality nominal features  
- ✅ When there's a strong relationship between the category and the target  
- ✅ Competition/Kaggle scenarios where squeezing performance matters

**When NOT to use:**  
- ❌ Small datasets (high risk of **target leakage** and overfitting)  
- ❌ Without proper regularization/smoothing  

**Critical Warning:** Always use **cross-validation** or **smoothing** to prevent target leakage! The `category_encoders` library handles this with built-in regularization.

In [ ]:
# Target Encoding — Using category_encoders (with regularization)

# First, create a numeric target
target = (df['Attrition'] == 'Yes').astype(int)

# Method 1: Manual (RISKY — prone to leakage)
mean_map = df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean()).to_dict()
df_target = df[['Department']].copy()
df_target['Dept_TargetEncoded_Manual'] = df['Department'].map(mean_map).round(4)

print('Manual Target Encoding (Attrition Rate by Department):')
print('-'*55)
for dept, rate in sorted(mean_map.items(), key=lambda x: -x[1]):
    print(f'  {dept:20s} -> Attrition Rate: {rate:.4f}')

print()

# Method 2: Using category_encoders (RECOMMENDED — has smoothing/regularization)
target_enc = ce.TargetEncoder(cols=['Department'], smoothing=1.0)
df_target['Dept_TargetEncoded_Regularized'] = target_enc.fit_transform(
    df[['Department']], target
)['Department'].round(4)

print('Comparison — Manual vs Regularized:')
df_target.drop_duplicates('Department').sort_values('Dept_TargetEncoded_Manual', ascending=False)

### 3.6 Leave-One-Out (LOO) Encoding

**What it does:** Similar to Target Encoding but excludes the **current row** when calculating the mean. This reduces target leakage.

**When to use:**  
- ✅ Same scenarios as Target Encoding but with better leakage protection  
- ✅ When you can't use cross-validation-based target encoding

**When NOT to use:**  
- ❌ Categories with very few samples (the leave-one-out mean becomes unstable)

In [ ]:
# Leave-One-Out Encoding
loo_enc = ce.LeaveOneOutEncoder(cols=['Department'])
df_loo = loo_enc.fit_transform(df[['Department']], target)
df_loo.columns = ['Dept_LOO_Encoded']

print('Leave-One-Out Encoding:')
print('Each row gets the mean target of its category EXCLUDING itself.')
print()

# Show a few Engineering rows — each will have slightly different values
comparison = pd.DataFrame({
    'Department': df['Department'],
    'Attrition': df['Attrition'],
    'LOO_Encoded': df_loo['Dept_LOO_Encoded'].round(4)
})
print('Engineering department rows (notice different values per row):')
comparison[comparison['Department'] == 'Engineering'].head(10)

### 3.7 Hashing Encoding

**What it does:** Applies a hash function to each category and maps it to a fixed number of columns.

**When to use:**  
- ✅ **Very high cardinality** features (1000+ categories, e.g., ZIP codes, product IDs)  
- ✅ When you need a fixed, predictable number of output columns  
- ✅ Online learning / streaming data where new categories may appear

**When NOT to use:**  
- ❌ When interpretability matters (hash collisions make it opaque)  
- ❌ Low cardinality features (OHE or Binary Encoding are better)

In [ ]:
# Hashing Encoding
hash_enc = ce.HashingEncoder(cols=['City'], n_components=4)  # Fixed 4 output columns
df_hash = hash_enc.fit_transform(df[['City']])

print(f'Hashing Encoding: 7 unique cities -> {df_hash.shape[1]} fixed columns')
print(f'Columns: {list(df_hash.columns)}')
print()
print('Note: Different cities may collide (map to same hash). This is the tradeoff.')
print()

result = pd.concat([df[['City']], df_hash], axis=1)
result.drop_duplicates('City').sort_values('City')

### 3.8 BaseN Encoding

**What it does:** Generalization of Binary Encoding. Converts label-encoded values to base-N representation. Binary Encoding is BaseN with base=2.

**When to use:**  
- ✅ When you want to control the tradeoff between number of columns and information retention  
- ✅ Base=2 (Binary), Base=3, Base=5, etc.

**When NOT to use:**  
- ❌ No specific disadvantage; it's a flexible generalization

In [ ]:
# BaseN Encoding — comparing different bases
for base in [2, 3, 5]:
    basen_enc = ce.BaseNEncoder(cols=['Department'], base=base)
    df_basen = basen_enc.fit_transform(df[['Department']])
    print(f'Base-{base} Encoding: 8 departments -> {df_basen.shape[1]} columns: {list(df_basen.columns)}')

print()
print('Observation: Higher base = fewer columns, but more values per column.')
print('Base-2 (Binary) is the most common choice.')

### 3.9 James-Stein Encoding

**What it does:** A Bayesian approach that **shrinks** the category mean toward the global mean. The shrinkage depends on the variance within each category — categories with fewer samples get shrunk more toward the global mean.

**When to use:**  
- ✅ When you want target encoding with **principled regularization**  
- ✅ Small categories that would otherwise be unreliable in target encoding  
- ✅ Continuous targets (works best here; for binary targets, use the `model='binary'` option)

**When NOT to use:**  
- ❌ When interpretability of the encoding values is critical

In [ ]:
# James-Stein Encoding
js_enc = ce.JamesSteinEncoder(cols=['Department'], model='binary')
df_js = js_enc.fit_transform(df[['Department']], target)
df_js.columns = ['Dept_JamesStein']

# Compare with Target Encoding
comparison = pd.DataFrame({
    'Department': df['Department'],
    'Target_Encoded': df_target['Dept_TargetEncoded_Manual'],
    'JamesStein_Encoded': df_js['Dept_JamesStein'].round(4)
}).drop_duplicates('Department').sort_values('Target_Encoded', ascending=False)

print('James-Stein vs Target Encoding:')
print('Notice how James-Stein shrinks extreme values toward the global mean.')
print(f'\nGlobal attrition rate: {target.mean():.4f}')
print()
comparison

---
## 4. Handling Ordinal Categorical Data

Ordinal data has a **meaningful order**. Examples: Education Level, Performance Rating, Satisfaction Level.

> **Golden Rule:** Always define the order explicitly. Never rely on alphabetical sorting!

### 4.1 OrdinalEncoder (sklearn) — Best for ML Pipelines

**What it does:** Maps categories to integers based on an **explicitly defined order**.

**When to use:**  
- ✅ Ordinal features in sklearn pipelines  
- ✅ When you need inverse_transform capability  
- ✅ Multiple ordinal columns with different orders

**When NOT to use:**  
- ❌ Nominal features (it will impose a false order)

In [ ]:
# Define the correct order for each ordinal feature
education_order = ['High School', 'Diploma', 'Bachelor', 'Master', 'PhD']
experience_order = ['0-2 years', '3-5 years', '6-10 years', '11-15 years', '16+ years']
performance_order = ['Poor', 'Below Average', 'Average', 'Good', 'Excellent']
satisfaction_order = ['Very Dissatisfied', 'Dissatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']
risk_order = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

# Fill NaN before encoding (OrdinalEncoder doesn't handle NaN by default)
df_ordinal = df[ordinal_cols].copy()
df_ordinal['Satisfaction_Level'] = df_ordinal['Satisfaction_Level'].fillna('Neutral')

# Apply OrdinalEncoder
ord_enc = OrdinalEncoder(categories=[
    education_order, experience_order, performance_order,
    satisfaction_order, risk_order
])

encoded_values = ord_enc.fit_transform(df_ordinal)
df_ord_encoded = pd.DataFrame(encoded_values, columns=[f'{c}_Encoded' for c in ordinal_cols])

# Show the mapping
print('OrdinalEncoder Mappings:')
print('='*60)
for col, order in zip(ordinal_cols, [education_order, experience_order, performance_order,
                                       satisfaction_order, risk_order]):
    print(f'\n{col}:')
    for i, cat in enumerate(order):
        print(f'  {cat:25s} -> {i}')

print()
df_ord_encoded.head(10)

### 4.2 Pandas Categorical with ordered=True — Best for EDA & Sorting

**What it does:** Converts a column to an **ordered categorical** dtype in pandas. This enables meaningful sorting, comparison operators (>, <), and memory efficiency.

**When to use:**  
- ✅ Exploratory Data Analysis (sorting, groupby respects order)  
- ✅ Visualization (plots will follow the defined order)  
- ✅ Memory optimization for large datasets

**When NOT to use:**  
- ❌ Direct input to sklearn models (they need numeric values)

In [ ]:
# Pandas Categorical with order
df_cat = df.copy()

# Convert Education_Level to ordered categorical
df_cat['Education_Level'] = pd.Categorical(
    df_cat['Education_Level'],
    categories=education_order,
    ordered=True
)

# Now we can do meaningful comparisons!
print('Employees with Education > Bachelor:')
print(f'Count: {(df_cat["Education_Level"] > "Bachelor").sum()}')
print()

# Sorting respects the order
print('Value counts (in correct order):')
print(df_cat['Education_Level'].value_counts().sort_index())
print()

# Memory efficiency
print(f'Memory as object dtype:     {df["Education_Level"].memory_usage(deep=True):,} bytes')
print(f'Memory as ordered category: {df_cat["Education_Level"].memory_usage(deep=True):,} bytes')

# Get the underlying codes (integer representation)
print(f'\nUnderlying codes: {df_cat["Education_Level"].cat.codes.unique()}')

In [ ]:
# Visualization benefit — bars appear in the correct logical order
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without ordering
df['Education_Level'].value_counts().plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Without Ordered Categorical\n(Alphabetical/Random Order)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# With ordering
df_cat['Education_Level'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('With Ordered Categorical\n(Logical Order: HS -> PhD)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.3 Manual Mapping (Dict-based) — Most Flexible

**What it does:** Use a Python dictionary to define exact numeric values for each category.

**When to use:**  
- ✅ When you need **custom spacing** (e.g., PhD should be weighted more than the gap between High School and Diploma)  
- ✅ When encoding doesn't follow simple sequential numbering  
- ✅ Quick and transparent

**When NOT to use:**  
- ❌ When you need inverse_transform or pipeline integration (OrdinalEncoder is better)

In [ ]:
# Manual Mapping with custom spacing

# Equal spacing (0, 1, 2, 3, 4)
edu_map_equal = {
    'High School': 0, 'Diploma': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4
}

# Custom spacing (reflecting real-world effort/value differences)
edu_map_custom = {
    'High School': 0, 'Diploma': 1, 'Bachelor': 3, 'Master': 5, 'PhD': 8
}

# Performance with custom weights
perf_map = {
    'Poor': 1, 'Below Average': 2, 'Average': 3, 'Good': 4, 'Excellent': 5
}

df_manual = df[['Education_Level', 'Performance_Rating']].copy()
df_manual['Edu_Equal_Spacing'] = df['Education_Level'].map(edu_map_equal)
df_manual['Edu_Custom_Spacing'] = df['Education_Level'].map(edu_map_custom)
df_manual['Perf_Mapped'] = df['Performance_Rating'].map(perf_map)

print('Manual Mapping — Equal vs Custom Spacing:')
print('='*65)
print(f'{"Category":20s} {"Equal Spacing":>15s} {"Custom Spacing":>15s}')
print('-'*55)
for cat in education_order:
    print(f'{cat:20s} {edu_map_equal[cat]:>15d} {edu_map_custom[cat]:>15d}')

print()
df_manual.head(10)

### 4.4 Label Encoding for Ordinal Data — Appropriate Here!

**What it does:** Same as Label Encoding in 3.1, but now it's **appropriate** because ordinal data HAS a natural order.

**When to use:**  
- ✅ Ordinal features — the imposed ordering matches reality  
- ✅ Quick encoding when categories sort alphabetically in the right order (rare but possible)

**Caveat:** LabelEncoder sorts alphabetically. If your categories don't sort correctly alphabetically (they usually don't!), use OrdinalEncoder or manual mapping instead.

In [ ]:
# Label Encoding on ordinal data — showing the alphabetical sorting problem
le = LabelEncoder()

le.fit(df['Performance_Rating'].dropna())
print('LabelEncoder on Performance_Rating (ALPHABETICAL order):')
for cls, code in zip(le.classes_, range(len(le.classes_))):
    print(f'  {cls:20s} -> {code}')

print()
print('⚠️  PROBLEM: "Average" got 0, "Below Average" got 1, "Excellent" got 2...')
print('   The alphabetical order does NOT match the logical order!')
print('   "Average" < "Below Average" < "Excellent" < "Good" < "Poor" — WRONG!')
print()
print('✅ SOLUTION: Use OrdinalEncoder or manual mapping where YOU define the order.')

---
## 5. Handling Binary Categorical Data

Binary features have exactly **2 categories**. These are the simplest to encode.

**Methods:**  
1. **Direct mapping** (0/1) — simplest and most common  
2. **LabelEncoder** — automatic  
3. **pd.get_dummies with drop_first** — creates exactly 1 column

In [ ]:
# Binary Encoding — All 3 methods produce the same result
df_binary = df[['Remote_Work', 'Attrition']].copy()

# Method 1: Direct mapping
df_binary['Remote_Work_Mapped'] = df['Remote_Work'].map({'Yes': 1, 'No': 0})
df_binary['Attrition_Mapped'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Method 2: LabelEncoder
le = LabelEncoder()
df_binary['Remote_Work_LE'] = le.fit_transform(df['Remote_Work'])
print(f'LabelEncoder mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Method 3: pd.get_dummies with drop_first
df_binary['Remote_Work_OHE'] = pd.get_dummies(df['Remote_Work'], drop_first=True).values

print()
print('All 3 methods compared:')
df_binary.head(10)

---
## 6. Handling Missing Values in Categorical Columns

Missing values in categorical features need special treatment before encoding.

**Common strategies:**  
1. **Mode imputation** — Replace with the most frequent category  
2. **Separate category** — Treat missing as a category "Unknown" or "Missing"  
3. **Forward/Backward fill** — For time-series ordered data  
4. **Model-based imputation** — Use another model to predict the missing category  
5. **Drop rows** — Only when missing percentage is very low (<1-2%)

In [ ]:
# Check missing values in our dataset
print('Missing values in categorical columns:')
for col in ordinal_cols + nominal_cols + binary_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        pct = n_missing / len(df) * 100
        print(f'  {col}: {n_missing} missing ({pct:.1f}%)')

print(f'\nSatisfaction_Level has {df["Satisfaction_Level"].isnull().sum()} missing values.')

In [ ]:
# Strategy 1: Mode Imputation
df_imputed = df.copy()
mode_val = df_imputed['Satisfaction_Level'].mode()[0]
df_imputed['Satisfaction_ModeImputed'] = df_imputed['Satisfaction_Level'].fillna(mode_val)
print(f'Strategy 1 - Mode Imputation: Filled with "{mode_val}"')

# Strategy 2: Separate "Missing" Category
df_imputed['Satisfaction_MissingCategory'] = df_imputed['Satisfaction_Level'].fillna('Unknown')
print(f'Strategy 2 - Missing as Category: Filled with "Unknown"')

# Strategy 3: KNN-based or Frequency-weighted Random Imputation
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
df_imputed['Satisfaction_SklearnImputed'] = imputer.fit_transform(
    df_imputed[['Satisfaction_Level']]
).ravel()
print(f'Strategy 3 - sklearn SimpleImputer (most_frequent): Filled with "{mode_val}"')

print()
# Show the rows that were originally missing
missing_mask = df['Satisfaction_Level'].isnull()
print('Rows that were originally missing:')
df_imputed[missing_mask][['Satisfaction_Level', 'Satisfaction_ModeImputed',
                           'Satisfaction_MissingCategory', 'Satisfaction_SklearnImputed']]

### When to Choose Which Imputation Strategy?

| Strategy | Use When | Avoid When |
|----------|----------|------------|
| **Mode** | Low % missing, dominant category exists | Data has roughly equal category frequencies |
| **"Unknown" Category** | Missingness itself could be informative | You're using distance-based models |
| **Drop Rows** | <1% missing, large dataset | Any significant missing % |
| **Model-based** | Complex relationships, resources available | Overkill for simple datasets |

---
## 7. Comparison Matrix — When to Use What

This is the **decision-making cheat sheet** for choosing the right encoding method.

In [ ]:
# Visual comparison matrix
comparison_data = {
    'Method': [
        'Label Encoding', 'One-Hot Encoding', 'One-Hot (drop_first)',
        'Binary Encoding', 'Frequency Encoding', 'Target Encoding',
        'Leave-One-Out', 'Hashing Encoding', 'BaseN Encoding',
        'James-Stein', 'OrdinalEncoder', 'Manual Mapping',
        'Pandas Categorical'
    ],
    'Data Type': [
        'Ordinal/Binary', 'Nominal', 'Nominal',
        'Nominal', 'Nominal', 'Nominal',
        'Nominal', 'Nominal', 'Nominal',
        'Nominal', 'Ordinal', 'Ordinal',
        'Ordinal'
    ],
    'Cardinality': [
        'Any', 'Low (2-15)', 'Low (2-15)',
        'Medium (10-100)', 'Any', 'High (50+)',
        'High (50+)', 'Very High (1000+)', 'Medium-High',
        'High (50+)', 'Any', 'Any',
        'Any'
    ],
    'Columns Created': [
        '1', 'n', 'n-1',
        'log2(n)', '1', '1',
        '1', 'Fixed k', 'logN(n)',
        '1', '1', '1',
        '1'
    ],
    'Best Model Type': [
        'Tree-based', 'Any', 'Linear/Logistic',
        'Any', 'Tree-based', 'Any (with care)',
        'Any (with care)', 'Any', 'Any',
        'Any', 'Any', 'Any',
        'EDA/Viz only'
    ],
    'Risk': [
        'False ordering', 'Dimensionality', 'Minimal',
        'Minor collision', 'Same-freq collision', 'Target leakage',
        'Unstable for small groups', 'Hash collision', 'Minor collision',
        'Minimal', 'Minimal', 'Minimal',
        'N/A'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print('ENCODING METHOD COMPARISON MATRIX')
print('='*100)
comparison_df

### Decision Flowchart

```
Is the feature ORDINAL (has natural order)?
├── YES → Use OrdinalEncoder or Manual Mapping
│         (Define the order explicitly!)
│
└── NO (Nominal)
    │
    ├── Is it BINARY (2 categories)?
    │   └── YES → Simple 0/1 mapping
    │
    ├── LOW cardinality (2-15)?
    │   └── One-Hot Encoding (drop_first for linear models)
    │
    ├── MEDIUM cardinality (15-100)?
    │   └── Binary Encoding or Target Encoding
    │
    └── HIGH cardinality (100+)?
        └── Target Encoding, Hashing, or Frequency Encoding
```

---
## 8. Full Pipeline Example with sklearn ColumnTransformer

This is how you'd combine everything into a **production-ready ML pipeline** that handles nominal, ordinal, and numerical features correctly.

In [ ]:
# Prepare the data
df_pipeline = df.copy()

# Handle missing values first
df_pipeline['Satisfaction_Level'] = df_pipeline['Satisfaction_Level'].fillna('Neutral')
df_pipeline['Salary'] = df_pipeline['Salary'].fillna(df_pipeline['Salary'].median())

# Define features and target
feature_cols = nominal_cols + ordinal_cols + ['Remote_Work'] + numerical_cols
X = df_pipeline[feature_cols]
y = (df_pipeline['Attrition'] == 'Yes').astype(int)

print(f'Features shape: {X.shape}')
print(f'Target distribution:\n{y.value_counts()}')
print(f'\nFeatures used: {feature_cols}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Define the order for each ordinal feature
ordinal_categories = [
    ['High School', 'Diploma', 'Bachelor', 'Master', 'PhD'],           # Education_Level
    ['0-2 years', '3-5 years', '6-10 years', '11-15 years', '16+ years'],  # Experience_Band
    ['Poor', 'Below Average', 'Average', 'Good', 'Excellent'],          # Performance_Rating
    ['Very Dissatisfied', 'Dissatisfied', 'Neutral', 'Satisfied', 'Very Satisfied'],  # Satisfaction
    ['Very Low', 'Low', 'Medium', 'High', 'Very High']                  # Risk_Appetite
]

# Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        # Nominal features → One-Hot Encoding
        ('nominal', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'),
         nominal_cols),

        # Ordinal features → OrdinalEncoder with explicit order
        ('ordinal', OrdinalEncoder(categories=ordinal_categories),
         ordinal_cols),

        # Binary feature → One-Hot with drop_first (= simple 0/1)
        ('binary', OneHotEncoder(drop='first', sparse_output=False),
         ['Remote_Work']),

        # Numerical features → StandardScaler
        ('numerical', StandardScaler(),
         numerical_cols)
    ],
    remainder='drop'
)

# Full pipeline with a classifier
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Pipeline created successfully!')
print()
print('Pipeline steps:')
print('  1. Nominal cols → OneHotEncoder (drop_first)')
print('  2. Ordinal cols → OrdinalEncoder (explicit order)')
print('  3. Binary cols  → OneHotEncoder (drop_first = 0/1)')
print('  4. Numerical    → StandardScaler')
print('  5. Classifier   → RandomForestClassifier')

In [ ]:
# Train-Test Split and Evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluate
print('MODEL EVALUATION')
print('='*50)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))

In [ ]:
# Inspect the transformed feature names
# Get feature names from the preprocessor
ohe_features = list(preprocessor.named_transformers_['nominal'].get_feature_names_out(nominal_cols))
ord_features = [f'{c}_OrdEncoded' for c in ordinal_cols]
bin_features = list(preprocessor.named_transformers_['binary'].get_feature_names_out(['Remote_Work']))
num_features = numerical_cols

all_features = ohe_features + ord_features + bin_features + num_features

print(f'Total features after preprocessing: {len(all_features)}')
print(f'  - From nominal OHE: {len(ohe_features)} features')
print(f'  - From ordinal encoding: {len(ord_features)} features')
print(f'  - From binary encoding: {len(bin_features)} features')
print(f'  - Numerical (scaled): {len(num_features)} features')
print()

# Feature importance from Random Forest
importances = pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances (After Encoding)', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 9. Key Takeaways & Best Practices

### The 7 Golden Rules

1. **Know your data type FIRST** — Is it Nominal, Ordinal, or Binary? This decides the method.

2. **Never impose order on nominal data** — Label Encoding on City/Department creates fake relationships.

3. **Always define ordinal order explicitly** — Don't rely on alphabetical sorting; use `categories=[...]` parameter.

4. **Match encoding to cardinality:**
   - Low (2-15 categories) → One-Hot Encoding
   - Medium (15-100) → Binary Encoding
   - High (100+) → Target, Hashing, or Frequency Encoding

5. **Match encoding to model type:**
   - Linear models → One-Hot (with `drop_first=True`)
   - Tree-based models → Almost any encoding works, but Label/Ordinal is memory-efficient
   - Neural Networks → One-Hot or learned embeddings

6. **Handle missing values BEFORE encoding** — Mode, "Unknown" category, or model-based imputation.

7. **Use sklearn Pipelines** — ColumnTransformer + Pipeline keeps everything reproducible and prevents data leakage.

### Quick Reference

| Scenario | Recommended Method |
|----------|-------------------|
| City (7 categories) + Logistic Regression | One-Hot Encoding (drop_first) |
| City (7 categories) + XGBoost | One-Hot or Binary Encoding |
| ZIP Code (10,000 categories) | Hashing or Target Encoding |
| Education Level (ordered) | OrdinalEncoder with explicit order |
| Yes/No feature | Simple 0/1 mapping |
| Performance Rating (ordered) | OrdinalEncoder or Manual Mapping |
| Product ID (50,000 categories) | Hashing Encoding or Embeddings |
| Feature where frequency matters | Frequency/Count Encoding |

---

**Notebook prepared by Prashant Nair | VNex (Vihaan NextGen Solutions)**  
**Reach out for corporate AI training at www.vnex.in**

---